# Superstore Sales Analysis
**GOAL~ Sql JOINs + Window Functions**

This notebook analyses superstore sales dataset and addresses the following concepts in business analysis:
- **JOINs** — combining data from multiple tables using a shared key
- **Window Functions** — ranking and comparing rows without collapsing them

**Business Insights being addressed are:**
1. Which customer segment + region combination is most profitable?
2. Which sub-categories rank highest within each product category?
3. How large is the sales gap between top and bottom sub-categories?

**Dataset:** Sample Superstore (Kaggle)  
**Tools:** Python · pandas · pandasql · JupyterLab

---
## Setup — Load Data

Loading dataset and importing the required libraries
 
 encoding='latin-1' is required for this CSV as it contains special characters that UTF-8 cannot read.

In [2]:
import pandas as pd
from pandasql import sqldf

df = pd.read_csv("../Sample - Superstore.csv", encoding='latin-1')

print(f'Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

Dataset shape: 9994 rows, 21 columns


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [2]:
# Simulate an orders table by splitting the table into two tables
orders_df = df[['Order ID', 'Customer ID', 'Sales', 'Profit', 
                'Quantity', 'Discount', 'Category']].copy()

# Simulate a customers table
customers_df = df[['Customer ID', 'Customer Name', 
                   'Segment', 'Region']].drop_duplicates().copy()

print(f'Orders table:    {orders_df.shape[0]} rows, {orders_df.shape[1]} columns')
print(f'Customers table: {customers_df.shape[0]} rows, {customers_df.shape[1]} columns')

print('\n--- Orders Table Preview ---')
print(orders_df.head(3).to_string())
print('\n--- Customers Table Preview ---')
print(customers_df.head(3).to_string())

Orders table:    9994 rows, 7 columns
Customers table: 2501 rows, 4 columns

--- Orders Table Preview ---
         Order ID Customer ID   Sales    Profit  Quantity  Discount         Category
0  CA-2016-152156    CG-12520  261.96   41.9136         2       0.0        Furniture
1  CA-2016-152156    CG-12520  731.94  219.5820         3       0.0        Furniture
2  CA-2016-138688    DV-13045   14.62    6.8714         2       0.0  Office Supplies

--- Customers Table Preview ---
  Customer ID    Customer Name    Segment Region
0    CG-12520      Claire Gute   Consumer  South
2    DV-13045  Darrin Van Huff  Corporate   West
3    SO-20335   Sean O'Donnell   Consumer  South


---
## Query 1 — INNER JOIN: Profitability by Segment and Region

**Business Insight:** Which customer segment and region combination drives the most profit?

**Importance:** Not all customer segments behave the same way across regions.
A Corporate segment might dominate in the West but underperform in the South.
This query gives management a precise view of where to focus sales efforts.

**SQL concepts used:**
- `INNER JOIN ... ON` — links orders to customers using the shared `Customer ID` key
- Table aliases `o` and `c` — shorthand so we don't repeat full table names
- `COUNT(DISTINCT ...)` — counts unique orders only, avoids double counting
- `GROUP BY` on two columns — creates one row per segment + region combination


In [3]:
q_join = """
SELECT 
    c.Segment,
    c.Region,
    COUNT(DISTINCT o."Order ID")    AS total_orders,
    ROUND(SUM(o.Sales), 2)          AS total_sales,
    ROUND(SUM(o.Profit), 2)         AS total_profit,
    ROUND(AVG(o.Discount) * 100, 2) AS avg_discount_pct
FROM orders_df o
INNER JOIN customers_df c 
    ON o."Customer ID" = c."Customer ID"
GROUP BY c.Segment, c.Region
ORDER BY total_profit DESC;
"""

result_join = sqldf(q_join, locals())
result_join

,Segment,Region,total_orders,total_sales,total_profit,avg_discount_pct
0,Consumer,West,2383,1072252.60,124206.43,15.83
1,Consumer,Central,2188,1015517.83,118651.29,16.39
2,Consumer,East,2304,1041384.11,118406.05,15.90
3,Consumer,South,1869,843850.74,106241.18,15.92
4,Corporate,West,1377,649918.13,91250.51,15.70
5,Corporate,East,1396,650272.63,83481.78,15.75
6,Corporate,Central,1236,557192.13,73973.97,16.77
7,Corporate,South,1070,501230.95,63648.17,15.64
8,Home Office,West,816,382268.24,54060.04,14.55
9,Home Office,Central,797,380670.03,53495.23,15.00


### Insight — Query 1: JOIN Results

**Business Insights:**
- Which Segment + Region combination has the **highest profit**?
The **Consumer segment in the West** region leads in total profit at $124206.43.

- Is there any combination with **high sales but negative profit**? What might explain this?
There isnt any high sales with negative profit.

- Which segment has the **highest average discount** and is that hurting their profit?
Notably, the **Home Office segment in the South** shows high discounting (15.15%) which correlates with its below-average profit margin — a pricing strategy worth reviewing.
**More observations:**
- Corporate Central has the highest discounting (16.77%) yet it's not the worst performer. That's actually a good sign — it means their discount strategy isn't destroying profit the way it did in sql foundations' query 5.
- Home Office is consistently the smallest segment across every region, lowest orders, lowest sales, lowest profit. But Home Office actually has a BETTER profit margin than Consumer despite being the smallest segment. That is a genuinely interesting business insight, the smallest customer group is the most efficient per dollar sold.
  
 


---
## Query 2 — RANK() Window Function: Sub-Category Rankings

**Business Insight:** Within each product category, which sub-categories are the top and bottom performers by sales?

**Importance:** A category like Furniture might look average overall, 
but one sub-category could be a star performer hiding inside a weak category. 
Window functions let us see this granular picture without losing the category context.

**SQL concepts used:**
- `RANK() OVER (PARTITION BY ... ORDER BY ...)` — ranks rows within each group
- `PARTITION BY Category` — resets the rank counter for each new category
- `ORDER BY SUM(Sales) DESC` — highest sales gets rank 1


In [4]:
q_window = """
SELECT 
    Category,
    "Sub-Category",
    ROUND(SUM(Sales), 2) AS total_revenue_usd,
    RANK() OVER (
        PARTITION BY Category 
        ORDER BY SUM(Sales) DESC
    ) AS rank_within_category
FROM df
GROUP BY Category, "Sub-Category"
ORDER BY Category, rank_within_category;
"""

result_window = sqldf(q_window, locals())
result_window

,Category,Sub-Category,total_sales,rank_within_category
0,Furniture,Chairs,328449.10,1
1,Furniture,Tables,206965.53,2
2,Furniture,Bookcases,114880.00,3
3,Furniture,Furnishings,91705.16,4
4,Office Supplies,Storage,223843.61,1
5,Office Supplies,Binders,203412.73,2
6,Office Supplies,Appliances,107532.16,3
7,Office Supplies,Paper,78479.21,4
8,Office Supplies,Supplies,46673.54,5
9,Office Supplies,Art,27118.79,6


### Insight — Query 2: Window Function Results

**Answer these questions from your results:**
- Which sub-category ranks **#1 in Technology**?

Within **Technology**, Phones rank #1 with 330007.05 in sales, nearly double the machines sub-category.  

- Which sub-category sits **last in Furniture** — and by how much does it trail the leader?

Within furniture, **Furninshings rank**  the last and trails off chairs by 328449.10 - 91705.16 = 236743.94

- Is there a sub-category that surprises you — either performing much better or worse than expected?
  
 Fasteners within Office Supplies are performing surprisingly low with the sales of only 3024.28 which raises the question that should the business even restock fasteners? Another possibility is that, they can be sold in volume but 



---
## Query 3 — LAG() Window Function: Sales Gap Analysis

**Business Insight:** How large is the sales gap between each sub-category and the one ranked above it?

**Impotance:** A huge gap between rank #1 and rank #2 signals market concentration — 
the business is heavily dependent on one product line. A small gap means competition 
is close and rankings could shift with modest investment.

**SQL concepts used:**
- `LAG()` — accesses the value from the **previous row** in the window
- `SUM(Sales) - LAG(SUM(Sales))` — subtracts the row above to get the gap
- The first row in each partition returns `NULL` for LAG (nothing above it)

In time series work, LAG becomes month-over-month or year-over-year growth — 
one of the most used patterns in business analytics.

In [3]:
q_lag = """
SELECT 
    Category,
    "Sub-Category",
    ROUND(SUM(Sales), 2) AS total_sales,
    RANK() OVER (
        PARTITION BY Category 
        ORDER BY SUM(Sales) DESC
    ) AS sales_rank,
    ROUND(SUM(Sales) - LAG(SUM(Sales)) OVER (
        PARTITION BY Category 
        ORDER BY SUM(Sales) DESC
    ), 2) AS sales_gap_from_above
FROM df
GROUP BY Category, "Sub-Category"
ORDER BY Category, sales_rank;
"""

result_lag = sqldf(q_lag, locals())
result_lag

,Category,Sub-Category,total_sales,sales_rank,sales_gap_from_above
0,Furniture,Chairs,328449.10,1,NaN
1,Furniture,Tables,206965.53,2,-121483.57
2,Furniture,Bookcases,114880.00,3,-92085.54
3,Furniture,Furnishings,91705.16,4,-23174.83
4,Office Supplies,Storage,223843.61,1,NaN
5,Office Supplies,Binders,203412.73,2,-20430.88
6,Office Supplies,Appliances,107532.16,3,-95880.57
7,Office Supplies,Paper,78479.21,4,-29052.95
8,Office Supplies,Supplies,46673.54,5,-31805.67
9,Office Supplies,Art,27118.79,6,-19554.75


### Insight — Query 3: LAG Results


**Answer these questions from your results:**
- Which category has the **largest gap** between rank #1 and rank #2 sub-category?
  In **Technology**, the gap between Phones (#1) and Machines (#2) is $ -140768.42 
- What does a large gap tell you about business risk?
  It represents a significant concentration risk. If phone sales declined, 
no other sub-category is large enough to compensate. Large gap = over-reliant on one product = high risk. Small gap = competition is close = more resilient.
- Which category has the **most evenly distributed** sales across sub-categories?
  **Office Supplies** shows the most balanced distribution, making it the most resilient category overall.

### other observations:
- Furniture's gap between #3 Bookcases and #4 Furnishings is only 23k very close.But the gap from #1 Chairs to #2 Tables is 121k. This means Furniture is top heavy
-  Chairs dominate but the rest are clustered together at the bottom.



---

### Key business findings
1. The Consumer segment dominates revenue across all regions, but Home Office 
   customers are the most profitable per dollar sold — making them an 
   undervalued segment worth targeting more aggressively.

2. Technology is the highest risk category — Phones alone drive 330k dollars in revenue 
   but the next sub-category (Machines) is 140k dollars behind. A drop in phone sales 
   would have no comparable replacement.

3. Office Supplies is the most resilient category with 9 sub-categories and 
   evenly distributed sales — Storage and Binders are close competitors, 
   meaning the category does not depend on any single product line.
